#Merge short-read per-donor h5ads into an all-donors object

Short-read source h5ads live outside this project under `../../CDKN2A_isoforms/` and use heterogeneous filenames per donor, so the donor → path map is hard-coded below. Edit `DONOR_SR_PATHS` to add/remove donors.

Per-donor preparation (mirrors the pattern in notebook 06_04–06_07):
- tag each row with its source `donor`
- map `popv_prediction` → `cell_ontology_class` via `csvs/popv_to_ts_celltype_mapping.csv`
- map `cell_ontology_class` → `broad_cell_class` via `csvs/ts_celltypes.csv`
- build `sample_id_10X_barcode` = `sample_id___10X_barcode` for cross-platform barcode matching with PacBio

Concatenates with `index_unique="_"` keyed by donor so barcode collisions across donors don't merge.

In [1]:
import logging
logging.getLogger("fontTools").setLevel(logging.WARNING)

import os
_r = os.path.abspath(".")
while _r != os.path.dirname(_r) and not os.path.exists(os.path.join(_r, ".notebooks_root")):
    _r = os.path.dirname(_r)
os.chdir(_r if os.path.exists(os.path.join(_r, ".notebooks_root")) else "/oak/stanford/groups/quake/mmantri/group.quake/tabula_longread/notebooks")  # cd to notebooks/ root (marked .notebooks_root); relocation-proof

import os, gc
import scanpy as sc
import anndata as ad
import pandas as pd
import numpy as np
gc.enable()

In [2]:
# Per-donor short-read sources (minimal h5ads written by 01_11_TSP32-33_fix_labels.ipynb:
# popv_prediction has been remapped into cell_ontology_class + broad_cell_class
# and dropped; non-raw layers / scvi obsm,uns entries have been stripped).
DONOR_SR_PATHS = {
    "TSP1_30": "./../../tabula_sapiens/TabulaSapiens_subset_objects_V2/TSP1_30_filtered_200gene_2500UMI_donorassay_version2d_withmetadata_processed0_minimal.h5ad", 
    "TSP33": "./../../CDKN2A_isoforms/TSP33/TSP33only_Version1c_BatchCorrected_WholeCell_10X_2025-01-10_popv_annotated_minimal.h5ad",
    "TSP32": "./../../CDKN2A_isoforms/TSP32/TSP32only_Version1c_BatchCorrected_FullObject_2025-01-10_popv_annotated_minimal.h5ad",
}

OUT_PATH = "./../pacbio/h5ads/all_donors_shortread_anndata.h5ad"

POPV_TO_TS_CSV = "./../csvs/popv_to_ts_celltype_mapping.csv"
TS_CELLTYPES_CSV = "./../csvs/ts_celltypes.csv"

popv_to_ts = pd.read_csv(POPV_TO_TS_CSV)
popv_to_ts_map = dict(zip(popv_to_ts["popv_prediction"], popv_to_ts["ts_cell_ontology_class"]))
ts_celltypes = pd.read_csv(TS_CELLTYPES_CSV)
ts_broad_map = dict(zip(ts_celltypes["cell_ontology_class"], ts_celltypes["broad_cell_class"]))
print(f"popv_to_ts_map: {len(popv_to_ts_map)} entries")
print(f"ts_broad_map:   {len(ts_broad_map)} entries")

popv_to_ts_map: 151 entries
ts_broad_map:   180 entries


In [3]:
adatas = {}

for donor, path in DONOR_SR_PATHS.items():
    if not os.path.exists(path):
        print(f"[skip] {donor}: {path} not found")
        continue
    print(f"\n=== {donor} ===")
    a = sc.read_h5ad(path)
    print(f"  loaded: {a.shape}")

    # Tag rows with source donor (overwrite if a stale value is present so the
    # column is authoritative after merge).
    a.obs["donor"] = donor

    # cell_ontology_class / broad_cell_class are expected to be pre-baked in
    # the minimal h5ads (see 01_11_TSP32-33_fix_labels.ipynb). If popv_prediction is
    # still present (non-minimal input), remap it here as a fallback.
    if "popv_prediction" in a.obs.columns:
        print(f"  popv_prediction present — remapping to cell_ontology_class + broad_cell_class")
        a.obs["cell_ontology_class"] = (
            a.obs["popv_prediction"].map(popv_to_ts_map).fillna("unknown")
        )
        a.obs["broad_cell_class"] = (
            a.obs["cell_ontology_class"].map(ts_broad_map).fillna("unknown")
        )
    missing_cols = {"cell_ontology_class", "broad_cell_class"} - set(a.obs.columns)
    assert not missing_cols, (
        f"{donor}: missing obs columns {missing_cols} and no popv_prediction to derive them from"
    )

    # sample_id___10X_barcode for cross-platform matching
    if {"sample_id", "10X_barcode"}.issubset(a.obs.columns):
        a.obs["sample_id_10X_barcode"] = (
            a.obs["sample_id"].astype(str) + "___" + a.obs["10X_barcode"].astype(str)
        )
    else:
        missing = {"sample_id", "10X_barcode"} - set(a.obs.columns)
        print(f"  warning: missing obs columns {missing} — skipping sample_id_10X_barcode build")

    adatas[donor] = a

if not adatas:
    raise RuntimeError("No donor h5ads were loaded — check DONOR_SR_PATHS.")

merged = ad.concat(adatas, join="outer", merge="first", index_unique="_")
print(f"\nmerged: {merged.shape}")

merged.obs['CDKN2A+ MKI67-'] = np.where(((merged[:,['CDKN2A']].to_df()["CDKN2A"]>0.0) & (merged[:,['MKI67']].to_df()["MKI67"] == 0.0)), 'Positive', 'Negative')
merged.obs['CDKN2A+ MKI67-'].value_counts()

os.makedirs(os.path.dirname(OUT_PATH), exist_ok=True)
merged.write_h5ad(OUT_PATH)
print(f"saved: {OUT_PATH}")

del adatas, merged
gc.collect()


=== TSP1_30 ===
  loaded: (1136218, 61806)

=== TSP33 ===
  loaded: (127794, 60606)

=== TSP32 ===
  loaded: (5404, 60606)

merged: (1269416, 61806)
saved: ./../pacbio/h5ads/all_donors_shortread_anndata.h5ad


0

In [2]:
merged = sc.read_h5ad("./../pacbio/h5ads/all_donors_shortread_anndata.h5ad")
merged

AnnData object with n_obs × n_vars = 1269416 × 61806
    obs: '10X_barcode', '10X_run', 'age', 'ambient_removal', 'anatomical_position', 'assay', 'broad_cell_class', 'cdna_plate', 'cdna_well', 'cell_ontology_class', 'cell_ontology_id', 'compartment', 'donor', 'donor_assay', 'donor_method', 'donor_tissue', 'donor_tissue_assay', 'ethnicity', 'free_annotation', 'library_plate', 'manually_annotated', 'method', 'n_genes_by_counts', 'notes', 'old_index', 'pct_counts_ercc', 'pct_counts_mt', 'replicate', 'sample_id', 'sample_number', 'sex', 'tissue', 'total_counts', 'total_counts_ercc', 'total_counts_mt', 'sample_id_10X_barcode', 'CDKN2A+ MKI67-'
    var: 'ensembl_id', 'gene_symbol', 'genome', 'mt', 'ercc', 'n_cells_by_counts', 'mean_counts', 'pct_dropout_by_counts', 'total_counts', 'mean', 'std'
    layers: 'raw_counts'

In [ ]:
# merged.obs.to_csv("./../csvs/all_donors_shortread_anndata_metadata.csv")

In [7]:
# --- Subset the already-saved merged object to the pacbio sample sheet's SR_sample_id,
#     keeping ALL cells of the xGen-pulldown donors (TSP21/25/27/33). Operates on `merged`
#     read from OUT_PATH above and overwrites OUT_PATH with the subset. ---
KEEP_ALL_DONORS = {"TSP21", "TSP25", "TSP27", "TSP33"}
KEEP_ASSAYS = {"10X_3Prime_v3.1", "WholeCell_10X_3Prime_v3.1"}
pacbio_sr_ids = set(pd.read_csv("./../csvs/sample_metadata.csv")["SR_sample_id"].dropna().astype(str)) - {"No match found"}

fine_donor = merged.obs["sample_id"].astype(str).str.split("_").str[0]   # fine donor from sample_id prefix
fine_assay = merged.obs["assay"]
keep = (fine_donor.isin(KEEP_ALL_DONORS) & fine_assay.isin(KEEP_ASSAYS)) | merged.obs["sample_id"].astype(str).isin(pacbio_sr_ids)
print(f"Subset merged: {merged.n_obs:,} -> {int(keep.sum()):,} cells "
      f"(keep-all donors {sorted(KEEP_ALL_DONORS)} + {len(pacbio_sr_ids)} pacbio SR samples)")
merged = merged[keep.values].copy()

# Export the LR-matched short-read cell metadata (obs of the subset object)
LR_META_CSV = "./../csvs/all_donors_shortread_LR_matched_anndata_metadata.csv"
merged.obs.to_csv(LR_META_CSV)
print(f"saved LR-matched metadata: {LR_META_CSV} ({merged.n_obs:,} cells)")

Subset merged: 1,269,416 -> 634,764 cells (keep-all donors ['TSP21', 'TSP25', 'TSP27', 'TSP33'] + 57 pacbio SR samples)


In [8]:
OUT_PATH = "./../pacbio/h5ads/all_donors_shortread_LR_matched_anndata.h5ad"
merged.write_h5ad(OUT_PATH)
print(f"saved subset: {OUT_PATH}")
merged

saved subset: ./../pacbio/h5ads/all_donors_shortread_LR_matched_anndata.h5ad


AnnData object with n_obs × n_vars = 634764 × 61806
    obs: '10X_barcode', '10X_run', 'age', 'ambient_removal', 'anatomical_position', 'assay', 'broad_cell_class', 'cdna_plate', 'cdna_well', 'cell_ontology_class', 'cell_ontology_id', 'compartment', 'donor', 'donor_assay', 'donor_method', 'donor_tissue', 'donor_tissue_assay', 'ethnicity', 'free_annotation', 'library_plate', 'manually_annotated', 'method', 'n_genes_by_counts', 'notes', 'old_index', 'pct_counts_ercc', 'pct_counts_mt', 'replicate', 'sample_id', 'sample_number', 'sex', 'tissue', 'total_counts', 'total_counts_ercc', 'total_counts_mt', 'sample_id_10X_barcode', 'CDKN2A+ MKI67-'
    var: 'ensembl_id', 'gene_symbol', 'genome', 'mt', 'ercc', 'n_cells_by_counts', 'mean_counts', 'pct_dropout_by_counts', 'total_counts', 'mean', 'std'
    layers: 'raw_counts'

In [9]:
# merged.obs.to_csv("./../csvs/all_donors_shortread_LR_matched_anndata_metadata.csv")